In [28]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Số lượng GPU: {torch.cuda.device_count()}")
else:
    print("Không tìm thấy GPU. Hãy kiểm tra cài đặt Settings.")

GPU: Tesla T4
Số lượng GPU: 2


In [29]:
import polars as pl
from pathlib import Path

In [30]:
import os
data_path="/kaggle/input/datasets/radek1/otto-train-and-test-data-for-local-validation"
print(os.listdir(data_path))

['test_labels.parquet', 'id2type.pkl', 'train.parquet', 'type2id.pkl', 'test.parquet']


In [33]:
data_train=pl.read_parquet(Path(data_path) / "train.parquet")
data_test=pl.read_parquet(Path(data_path) / "test.parquet")

In [31]:
data=pl.read_parquet(Path(data_path) / "test_labels.parquet")

In [35]:
data.head(5)

session,type,ground_truth
i64,str,list[i64]
11098528,"""clicks""",[1679529]
11098528,"""carts""",[1199737]
11098528,"""orders""","[990658, 950341, … 1033148]"
11098529,"""clicks""",[1105029]
11098530,"""orders""",[409236]


In [32]:
type(data_train)

polars.dataframe.frame.DataFrame

In [33]:
data_train.head(5)

session,aid,ts,type
i32,i32,i32,u8
0,1517085,1659304800,0
0,1563459,1659304904,0
0,1309446,1659367439,0
0,16246,1659367719,0
0,1781822,1659367871,0


In [34]:
data_test.head(5)

session,aid,ts,type
i32,i32,i32,u8
11098528,11830,1661119200,0
11098529,1105029,1661119200,0
11098530,264500,1661119200,0
11098530,264500,1661119288,0
11098530,409236,1661119369,0


In [35]:
#Kiểm tra xem train và test có session nào trùng ko : session tồn tại ở cả tập train và tập test.
import polars as pl

# 1. Lấy danh sách session duy nhất của từng bộ dưới dạng Series
train_sessions = data_train["session"].unique()
test_sessions = data_test["session"].unique()

# 2. Tìm các session xuất hiện ở CẢ HAI bộ dữ liệu (Giao nhau)
# Chuyển sang tập hợp (set) của Python để tính toán giao nhau nhanh nhất
trung_sessions = set(train_sessions).intersection(set(test_sessions))

# 3. In kết quả kiểm tra
print("=" * 60)
print("KIỂM TRA TRÙNG LẶP ID SESSION GIỮA TRAIN VÀ TEST")
print("=" * 60)
print(f"Số lượng session duy nhất trong TRAIN: {len(train_sessions):,}")
print(f"Số lượng session duy nhất trong TEST:  {len(test_sessions):,}")
print("-" * 60)
print(f"SỐ LƯỢNG SESSION BỊ TRÙNG LẶP GIỮA HAI BỘ: {len(trung_sessions):,}")
print("=" * 60)

# Nếu có trùng, in thử ra 5 ID đầu tiên để quan sát
if len(trung_sessions) == 0:
    print("Không có session nào bị trùng giữa TRAIN và TEST.")
else:
    print(f"Có {len(trung_sessions):,} session bị trùng giữa TRAIN và TEST.")
    print(f"Danh sach cac session bi trung: {list(trung_sessions)[:5]}")

KIỂM TRA TRÙNG LẶP ID SESSION GIỮA TRAIN VÀ TEST
Số lượng session duy nhất trong TRAIN: 11,098,528
Số lượng session duy nhất trong TEST:  1,801,251
------------------------------------------------------------
SỐ LƯỢNG SESSION BỊ TRÙNG LẶP GIỮA HAI BỘ: 0
Không có session nào bị trùng giữa TRAIN và TEST.


In [36]:
# 1. Chuyển đổi timestamp và trích xuất giờ, ngày trong tuần
# Unix Timestamp (Epoch Time)
'''
Trong hệ thống máy tính, việc lưu trữ ngày tháng dưới dạng chữ 
(ví dụ: "2026-05-23 10:35:00") rất tốn bộ nhớ và khó tính toán (ví dụ như khi muốn cộng, trừ thời gian).

Vì vậy, các hệ thống thường quy đổi thời gian thành một con số duy nhất. 
Con số này chính là tổng số giây (hoặc mili giây) trôi qua kể từ mốc thời gian gốc
 là 00:00:00 ngày 01 tháng 01 năm 1970 UTC (gọi là thời điểm Epoch).
'''
data_train = data_train.with_columns(
    [
        # Chuyển ts (mili giây) sang kiểu Datetime
        pl.from_epoch("ts", time_unit="ms").alias("timestamp"),
        # Trích xuất giờ (0 - 23)
        pl.from_epoch("ts", time_unit="ms").dt.hour().alias("hour"),
        # Trích xuất ngày trong tuần (1 = Thứ 2, 7 = Chủ Nhật)
        pl.from_epoch("ts", time_unit="ms").dt.weekday().alias("day_of_week"),
    ]
)
data_test = data_test.with_columns(
    [
        # Chuyển ts (mili giây) Dang Unix  sang kiểu Datetime
        pl.from_epoch("ts", time_unit="ms").alias("timestamp"),
        # Trích xuất giờ (0 - 23)
        pl.from_epoch("ts", time_unit="ms").dt.hour().alias("hour"),
        # Trích xuất ngày trong tuần (1 = Thứ 2, 7 = Chủ Nhật)
        pl.from_epoch("ts", time_unit="ms").dt.weekday().alias("day_of_week"),
    ]
)
# 2. Không cần cache() chủ động vì Polars tối ưu hóa bộ nhớ rất tốt
# (Nếu bạn dùng LazyFrame, bạn chỉ cần gọi .collect() để thực thi)

# 3. Hiển thị 5 dòng đầu tiên
print(data_train.head(5))
print(data_test.head(5))

shape: (5, 7)
┌─────────┬─────────┬────────────┬──────┬─────────────────────────┬──────┬─────────────┐
│ session ┆ aid     ┆ ts         ┆ type ┆ timestamp               ┆ hour ┆ day_of_week │
│ ---     ┆ ---     ┆ ---        ┆ ---  ┆ ---                     ┆ ---  ┆ ---         │
│ i32     ┆ i32     ┆ i32        ┆ u8   ┆ datetime[ms]            ┆ i8   ┆ i8          │
╞═════════╪═════════╪════════════╪══════╪═════════════════════════╪══════╪═════════════╡
│ 0       ┆ 1517085 ┆ 1659304800 ┆ 0    ┆ 1970-01-20 04:55:04.800 ┆ 4    ┆ 2           │
│ 0       ┆ 1563459 ┆ 1659304904 ┆ 0    ┆ 1970-01-20 04:55:04.904 ┆ 4    ┆ 2           │
│ 0       ┆ 1309446 ┆ 1659367439 ┆ 0    ┆ 1970-01-20 04:56:07.439 ┆ 4    ┆ 2           │
│ 0       ┆ 16246   ┆ 1659367719 ┆ 0    ┆ 1970-01-20 04:56:07.719 ┆ 4    ┆ 2           │
│ 0       ┆ 1781822 ┆ 1659367871 ┆ 0    ┆ 1970-01-20 04:56:07.871 ┆ 4    ┆ 2           │
└─────────┴─────────┴────────────┴──────┴─────────────────────────┴──────┴─────────────┘
shape: 

In [37]:
pl.Config.set_tbl_rows(20)

# =====================================================================
# 1. TRÍCH XUẤT VÀ ĐẾM SỐ LƯỢNG SẢN PHẨM TRONG BỘ TRAIN
# =====================================================================
train_aid_counts = (
    data_train.group_by("aid")
    .agg(pl.len().alias("so_luot_tuong_tac"))
    .sort("so_luot_tuong_tac", descending=True)
)

# =====================================================================
# 2. TRÍCH XUẤT VÀ ĐẾM SỐ LƯỢNG SẢN PHẨM TRONG BỘ TEST
# =====================================================================
test_aid_counts = (
    data_test.group_by("aid")
    .agg(pl.len().alias("so_luot_tuong_tac"))
    .sort("so_luot_tuong_tac", descending=True)
)

# =====================================================================
# 3. TÍNH TỔNG SỐ LƯỢNG SẢN PHẨM DUY NHẤT (UNIQUE ITEMS)
# =====================================================================
tong_aid_train = train_aid_counts.height
tong_aid_test = test_aid_counts.height

# Tính số sản phẩm xuất hiện ở cả 2 bộ (Giao nhau)
giao_aid = len(set(train_aid_counts["aid"]).intersection(set(test_aid_counts["aid"])))

# =====================================================================
# 4. IN KẾT QUẢ THỐNG KÊ TỔNG QUAN
# =====================================================================
print("=" * 60)
print("THỐNG KÊ SỐ LƯỢNG SẢN PHẨM (AID) TRONG TRAIN & TEST")
print("=" * 60)
print(f"Tổng số sản phẩm duy nhất trong bộ TRAIN: {tong_aid_train:,}")
print(f"Tổng số sản phẩm duy nhất trong bộ TEST:  {tong_aid_test:,}")
print(f"Số sản phẩm ở bộ TEST cũng có mặt trong TRAIN: {giao_aid:,}")
print(f"-> Tỷ lệ bao phủ (Coverage): {(giao_aid / tong_aid_test) * 100:.2f}%")
print("-" * 60)

print("\n[TOP 5 SẢN PHẨM TƯƠNG TÁC NHIỀU NHẤT TRONG TRAIN]:")
print(train_aid_counts.head(5))

print("\n[TOP 5 SẢN PHẨM TƯƠNG TÁC NHIỀU NHẤT TRONG TEST]:")
print(test_aid_counts.head(5))
print("=" * 60)

THỐNG KÊ SỐ LƯỢNG SẢN PHẨM (AID) TRONG TRAIN & TEST
Tổng số sản phẩm duy nhất trong bộ TRAIN: 1,825,499
Tổng số sản phẩm duy nhất trong bộ TEST:  874,852
Số sản phẩm ở bộ TEST cũng có mặt trong TRAIN: 856,067
-> Tỷ lệ bao phủ (Coverage): 97.85%
------------------------------------------------------------

[TOP 5 SẢN PHẨM TƯƠNG TÁC NHIỀU NHẤT TRONG TRAIN]:
shape: (5, 2)
┌─────────┬───────────────────┐
│ aid     ┆ so_luot_tuong_tac │
│ ---     ┆ ---               │
│ i32     ┆ u32               │
╞═════════╪═══════════════════╡
│ 29735   ┆ 92407             │
│ 1460571 ┆ 91778             │
│ 1733943 ┆ 88723             │
│ 108125  ┆ 88126             │
│ 832192  ┆ 73222             │
└─────────┴───────────────────┘

[TOP 5 SẢN PHẨM TƯƠNG TÁC NHIỀU NHẤT TRONG TEST]:
shape: (5, 2)
┌─────────┬───────────────────┐
│ aid     ┆ so_luot_tuong_tac │
│ ---     ┆ ---               │
│ i32     ┆ u32               │
╞═════════╪═══════════════════╡
│ 485256  ┆ 36584             │
│ 1460571 ┆ 8008   

In [38]:
import polars as pl

# 1. Định nghĩa ID session cần kiểm tra
target_session = 1517085

# 2. Lọc dữ liệu của session đó từ data_train
session_data = data_train.filter(pl.col("session") == target_session)

# 3. Kiểm tra và in kết quả kèm điều kiện
print("=" * 65)
print(f"KẾT QUẢ KIỂM TRA SESSION ID: {target_session}")
print("=" * 65)

if session_data.height > 0:
    print(f"Tìm thấy: Có tồn tại session {target_session} trong bộ data_train.")
    print(f"Tổng số lượt tương tác của session này: {session_data.height} dòng.")
    print("-" * 65)
    print("Dưới đây là toàn bộ chuỗi hành vi của session:")
    
    # Sắp xếp theo ts tăng dần để xem chuỗi hành vi theo đúng thứ tự thời gian tuyến tính
    print(session_data.sort("ts"))
else:
    print(f"Không tìm thấy: Session {target_session} KHÔNG tồn tại trong bộ data_train.")

print("=" * 65)

KẾT QUẢ KIỂM TRA SESSION ID: 1517085
Tìm thấy: Có tồn tại session 1517085 trong bộ data_train.
Tổng số lượt tương tác của session này: 2 dòng.
-----------------------------------------------------------------
Dưới đây là toàn bộ chuỗi hành vi của session:
shape: (2, 7)
┌─────────┬─────────┬────────────┬──────┬─────────────────────────┬──────┬─────────────┐
│ session ┆ aid     ┆ ts         ┆ type ┆ timestamp               ┆ hour ┆ day_of_week │
│ ---     ┆ ---     ┆ ---        ┆ ---  ┆ ---                     ┆ ---  ┆ ---         │
│ i32     ┆ i32     ┆ i32        ┆ u8   ┆ datetime[ms]            ┆ i8   ┆ i8          │
╞═════════╪═════════╪════════════╪══════╪═════════════════════════╪══════╪═════════════╡
│ 1517085 ┆ 1032830 ┆ 1659436189 ┆ 0    ┆ 1970-01-20 04:57:16.189 ┆ 4    ┆ 2           │
│ 1517085 ┆ 1630262 ┆ 1659436226 ┆ 0    ┆ 1970-01-20 04:57:16.226 ┆ 4    ┆ 2           │
└─────────┴─────────┴────────────┴──────┴─────────────────────────┴──────┴─────────────┘


In [41]:
session_stats = data_train.group_by("session").agg(
    [
        pl.len().alias("so_tuong_tac"),
        pl.col("timestamp").min().alias("thoi_gian_dau"),
        pl.col("timestamp").max().alias("thoi_gian_cuoi"),
    ]
)
session_stats = session_stats.sort("session", descending=False)
print(session_stats.head(10))

shape: (10, 4)
┌─────────┬──────────────┬─────────────────────────┬─────────────────────────┐
│ session ┆ so_tuong_tac ┆ thoi_gian_dau           ┆ thoi_gian_cuoi          │
│ ---     ┆ ---          ┆ ---                     ┆ ---                     │
│ i32     ┆ u32          ┆ datetime[ms]            ┆ datetime[ms]            │
╞═════════╪══════════════╪═════════════════════════╪═════════════════════════╡
│ 0       ┆ 147          ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:25:03.727 │
│ 1       ┆ 27           ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:20:57.067 │
│ 2       ┆ 13           ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:16:17.379 │
│ 3       ┆ 226          ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:25:09.666 │
│ 4       ┆ 3            ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 04:55:04.900 │
│ 5       ┆ 15           ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:12:28.787 │
│ 6       ┆ 174          ┆ 1970-01-20 04:55:04.800 ┆ 1970-01-20 05:22:21.769 │
│ 7       ┆ 23           ┆ 1970-01-20